# 🎬 VideoScene — Analyse vidéo horodatée

Ce notebook analyse ta vidéo image par image et produit un script avec des timestamps précis.

## 👉 Comment utiliser :
1. Clique sur **Exécution → Tout exécuter** (ou appuie sur `Ctrl+F9`)
2. Entre ta clé API Google quand demandé
3. Uploade ta vidéo quand demandé
4. Attends la fin de l'analyse
5. Télécharge ton script TXT

---

In [ ]:
# ─── ÉTAPE 1 : Installation ───────────────────────────────────────────────────
print('Installation en cours...')
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
subprocess.run(['pip', 'install', 'google-genai', 'Pillow', '-q'], capture_output=True)
print('✅ Installation terminée')

In [ ]:
# ─── ÉTAPE 2 : Configuration ──────────────────────────────────────────────────
#@title ⚙️ Configuration { run: "auto" }

CLE_API_GOOGLE = "" #@param {type:"string", placeholder:"Colle ta clé AIza... ici"}
MODE = "Script narratif (exhaustif, 1 frame/5s)" #@param ["Résumé par scène (rapide, 1 frame/30s)", "Script narratif (exhaustif, 1 frame/5s)"]

if not CLE_API_GOOGLE:
    print('⚠️  Entre ta clé API Google dans le champ ci-dessus puis relance cette cellule.')
else:
    print(f'✅ Clé API configurée')
    print(f'✅ Mode sélectionné : {MODE}')

In [ ]:
# ─── ÉTAPE 3 : Upload de la vidéo ─────────────────────────────────────────────
from google.colab import files

print('📂 Sélectionne ta vidéo (MP4, MOV, AVI — max 15 minutes)...')
uploaded = files.upload()

if uploaded:
    VIDEO_PATH = list(uploaded.keys())[0]
    size_mb = len(uploaded[VIDEO_PATH]) / 1024 / 1024
    print(f'✅ Vidéo reçue : {VIDEO_PATH} ({size_mb:.1f} Mo)')
else:
    print('⚠️  Aucun fichier sélectionné.')

In [ ]:
# ─── ÉTAPE 4 : Analyse ────────────────────────────────────────────────────────
import re, json, time, shutil, os
from pathlib import Path
from google import genai
from PIL import Image

# Config selon le mode
if 'narratif' in MODE.lower():
    SCENE_SEUIL = 0.20
    INTERVALLE  = 5
    PROMPT = (
        'Tu analyses une image extraite dune vidéo.\n'
        'Décris de façon exhaustive et fluide tout ce que tu vois à lécran, '
        'comme un narrateur qui décrit un film en temps réel.\n'
        'Couvre : actions, mouvements, expressions, décors, angle de caméra, '
        'texte affiché, objets importants, transitions.\n'
        'NE TRANSCRIS PAS les dialogues ou paroles.\n'
        'Écris directement la narration en paragraphe continu, sans titre ni liste.'
    )
else:
    SCENE_SEUIL = 0.30
    INTERVALLE  = 30
    PROMPT = (
        'Décris précisément et de façon concise ce que tu vois dans cette frame vidéo : '
        'personnes, actions, objets importants, lieu, ambiance. Maximum 3 phrases.'
    )

# ── Durée de la vidéo
def get_duree(path):
    r = subprocess.run(
        ['ffprobe','-v','quiet','-print_format','json','-show_format', path],
        capture_output=True, text=True
    )
    return float(json.loads(r.stdout)['format']['duration'])

# ── Format timestamps
def ts_long(s):
    h,m = int(s//3600), int((s%3600)//60)
    return f'{h:02d}:{m:02d}:{s%60:06.3f}'

def ts_script(s):
    h,m,sec = int(s//3600), int((s%3600)//60), int(s%60)
    return f'[{h:02d}:{m:02d}:{sec:02d}]' if h else f'[{m:02d}:{sec:02d}]'

# ── Extraction des frames
def extraire_frames(video_path, seuil, intervalle):
    out = Path('/tmp/frames')
    if out.exists(): shutil.rmtree(out)
    out.mkdir()
    raw = out / 'raw'
    raw.mkdir()

    vf = (
        f"select='gt(scene,{seuil})+isnan(prev_selected_t)+"
        f"gte(t-prev_selected_t\\,{intervalle})',showinfo,"
        "scale=1280:720:force_original_aspect_ratio=decrease"
    )
    r = subprocess.run(
        ['ffmpeg','-y','-i', video_path,'-vf', vf,
         '-vsync','vfr','-q:v','2', str(raw/'%06d.jpg')],
        capture_output=True, text=True, timeout=300
    )

    timestamps = []
    for line in r.stderr.split('\n'):
        if 'pts_time' in line and 'showinfo' in line.lower():
            m = re.search(r'pts_time:([\d.]+)', line)
            if m: timestamps.append(float(m.group(1)))

    frames = sorted(raw.glob('*.jpg'))
    resultats = []
    for i, f in enumerate(frames):
        if i >= len(timestamps): break
        t = timestamps[i]
        dest = out / f'f_{t:012.4f}'.replace('.','_') + '.jpg'
        # workaround: rename with string concat
        dest = out / ('f_' + f'{t:012.4f}'.replace('.','_') + '.jpg')
        f.rename(dest)
        resultats.append({'path': str(dest), 'ts': t})

    shutil.rmtree(raw, ignore_errors=True)
    return sorted(resultats, key=lambda x: x['ts'])

# ── Analyse
duree = get_duree(VIDEO_PATH)
if duree > 15*60:
    print(f'⚠️  Vidéo trop longue ({ts_long(duree)}). Maximum 15 minutes.')
else:
    print(f'⏱️  Durée : {ts_long(duree)}')
    print(f'🔍 Extraction des frames ({INTERVALLE}s d\'intervalle)...')

    frames = extraire_frames(VIDEO_PATH, SCENE_SEUIL, INTERVALLE)
    print(f'📸 {len(frames)} instants à analyser')

    client = genai.Client(api_key=CLE_API_GOOGLE)
    resultats = []

    print('\n🤖 Analyse par Gemini en cours...\n')
    print('─' * 60)

    for i, frame in enumerate(frames):
        ts   = frame['ts']
        label = ts_script(ts)

        try:
            img  = Image.open(frame['path'])
            resp = client.models.generate_content(
                model='gemini-2.5-flash-lite',
                contents=[PROMPT, img]
            )
            texte = resp.text.strip()
            resultats.append({'ts': ts, 'label': label, 'texte': texte})

            # Affichage en direct
            print(f'{label} {texte[:120]}...' if len(texte) > 120 else f'{label} {texte}')
            print()

        except Exception as e:
            print(f'{label} ⚠️  Ignoré : {e}')

        # Quota gratuit : 15 req/min max
        if i < len(frames) - 1:
            time.sleep(4)

    print('─' * 60)
    print(f'\n✅ Analyse terminée — {len(resultats)} instants analysés')
    print('Lance la cellule suivante pour télécharger le script.')

In [ ]:
# ─── ÉTAPE 5 : Télécharger le script ──────────────────────────────────────────
from google.colab import files as colab_files

nom_base = Path(VIDEO_PATH).stem
nom_txt  = f'script_{nom_base}.txt'

entete = (
    f'SCRIPT NARRATIF — VideoScene\n'
    f'Vidéo  : {VIDEO_PATH}\n'
    f'Durée  : {ts_long(duree)}\n'
    f'Mode   : {MODE}\n'
    f'Scènes : {len(resultats)}\n'
    f'{"─" * 60}\n\n'
)

corps = '\n\n'.join(f"{r['label']} {r['texte']}" for r in resultats)

with open(nom_txt, 'w', encoding='utf-8') as f:
    f.write(entete + corps)

print(f'📄 Téléchargement de {nom_txt}...')
colab_files.download(nom_txt)
print('✅ Fichier téléchargé !')